[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C08_Training_Systems_Course/05_quantization_from_scratch/05_quantization_from_scratch.ipynb)

# 05 · 量化从零：int8/int4 与 GPTQ/AWQ 思想

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
下载**真实 GPT-2 权重矩阵**，从零实现对称/非对称 int8、per-channel、int4 打包，并测真实量化误差。

**你将完成：**
1. 用 HTTP range 请求从 safetensors 读一个真实 GPT-2 权重（无需 torch）
2. 对称（absmax）int8 量化/反量化 + 误差
3. per-tensor vs per-channel：误差差多少
4. int4 打包 + 简化版"保护重要列"（GPTQ/AWQ 思想）

> 数据：openai-community/gpt2 的 `h.0.attn.c_proj.weight`（768×768, 真实权重）。

## 0 · 下载真实 GPT-2 权重张量

In [ ]:
import os, json, struct, urllib.request
import numpy as np
np.set_printoptions(precision=4, suppress=True)
CACHE=os.path.expanduser("~/.training_systems_data"); os.makedirs(CACHE, exist_ok=True)

def load_gpt2_weight(name="h.0.attn.c_proj.weight"):
    cache=os.path.join(CACHE, f"gpt2_{name.replace('.','_')}.npy")
    if os.path.exists(cache): return np.load(cache)
    URL="https://huggingface.co/openai-community/gpt2/resolve/main/model.safetensors"
    def rng(s,e):
        req=urllib.request.Request(URL, headers={"Range":f"bytes={s}-{e}"})
        return urllib.request.urlopen(req, timeout=60).read()
    hlen=struct.unpack("<Q", rng(0,7))[0]
    hdr=json.loads(rng(8, 8+hlen-1))
    info=hdr[name]; s,e=info["data_offsets"]; base=8+hlen
    W=np.frombuffer(rng(base+s, base+e-1), dtype=np.float32).reshape(info["shape"]).copy()
    np.save(cache, W); return W

W = load_gpt2_weight()
print("真实 GPT-2 权重:", W.shape, "dtype", W.dtype)
print(f"  范围 [{W.min():.3f}, {W.max():.3f}]  均值 {W.mean():.4f}  std {W.std():.4f}")
print(f"  近似零均值对称分布 -> 适合对称量化")

## 1 · 对称量化（absmax int8）

s = max|W|/127，q=round(W/s)，反量化 Ŵ=q·s。测相对误差。

In [ ]:
def quant_symmetric(W):
    s = np.abs(W).max()/127
    q = np.round(W/s).astype(np.int8)
    return q, s
def dequant_symmetric(q, s): return q.astype(np.float32)*s

q, s = quant_symmetric(W)
Wq = dequant_symmetric(q, s)
err = np.abs(W-Wq).mean()/np.abs(W).mean()
print(f"int8 对称量化: scale={s:.6f}")
print(f"平均相对误差 = {err:.4%}")
print(f"显存: fp32 {W.nbytes/1024:.0f}KB -> int8 {q.nbytes/1024:.0f}KB (1/4) + 1个scale")
print("\n注意: 这块真实权重有离群值(范围±3.3), per-tensor absmax 被它撑大 scale,")
print("误差竟有 ~9% 而非课本说的 <1% —— 这正是 per-channel(第3节)与离群值处理的动机。")

## 2 · 非对称量化（zero-point uint8）

对不对称数据更省范围。这里在 ReLU 后的"激活"上演示（全正）。

In [ ]:
def quant_asymmetric(X):
    lo, hi = X.min(), X.max()
    s = (hi-lo)/255; z = np.round(-lo/s)
    q = np.clip(np.round(X/s)+z, 0, 255).astype(np.uint8)
    return q, s, z
def dequant_asymmetric(q, s, z): return (q.astype(np.float32)-z)*s

act = np.maximum(0, W @ np.random.default_rng(0).normal(size=(W.shape[1],1))).ravel()  # 模拟 ReLU 激活(全正)
q,s,z = quant_asymmetric(act); aq=dequant_asymmetric(q,s,z)
print(f"非对称量化激活(全正): zero-point z={z:.0f}, scale={s:.5f}")
print(f"相对误差 = {np.abs(act-aq).mean()/(np.abs(act).mean()+1e-9):.4%}")
print("=> 非对称把整个 [0,max] 映射到 [0,255]，不浪费负半轴")

## 3 · per-tensor vs per-channel

per-channel（每列一个 scale）让离群列不拖累其他列，误差更小。

In [ ]:
def quant_per_channel(W):  # 每列一个 scale
    s = np.abs(W).max(0, keepdims=True)/127
    q = np.round(W/s).astype(np.int8)
    return q, s
def err_of(W, Wq): return np.abs(W-Wq).mean()/np.abs(W).mean()

q_t, s_t = quant_symmetric(W); err_t = err_of(W, dequant_symmetric(q_t,s_t))
q_c, s_c = quant_per_channel(W); err_c = err_of(W, q_c.astype(np.float32)*s_c)
print(f"per-tensor  相对误差 = {err_t:.4%}  (1 个 scale)")
print(f"per-channel 相对误差 = {err_c:.4%}  ({W.shape[1]} 个 scale)")
print(f"=> per-channel 误差更小 ({err_t/err_c:.1f}x)，代价是多存 {W.shape[1]} 个 scale")

## 4 · int4 打包

两个 int4 塞进一个字节，显存是 fp16 的 1/4。

In [ ]:
def quant_int4_pack(W):
    s = np.abs(W).max()/7                  # int4 对称: [-7,7]
    q = np.clip(np.round(W/s), -7, 7).astype(np.int8).ravel()
    qu = (q + 8).astype(np.uint8)          # 移到 [1,15] 便于打包
    if len(qu)%2: qu=np.append(qu,8)
    packed = (qu[0::2] << 4) | qu[1::2]    # 两个塞一字节
    return packed.astype(np.uint8), s, W.shape
def dequant_int4(packed, s, shape):
    hi = (packed >> 4) & 0xF; lo = packed & 0xF
    qu = np.empty(len(packed)*2, dtype=np.uint8); qu[0::2]=hi; qu[1::2]=lo
    q = qu[:np.prod(shape)].astype(np.int8) - 8
    return (q.astype(np.float32)*s).reshape(shape)

packed, s4, shp = quant_int4_pack(W)
W4 = dequant_int4(packed, s4, shp)
print(f"int4 打包: {W.nbytes/1024:.0f}KB(fp32) -> {packed.nbytes/1024:.0f}KB (1/8 of fp32, 1/4 of fp16)")
print(f"int4 相对误差 = {err_of(W,W4):.2%}  (只有16级，误差明显大于int8)")

## 5 · 量化误差的理论值 s²/12

均匀量化的误差可以建模成均匀分布 $U(-s/2, s/2)$（$s$ 是量化步长），方差 $= s^2/12$、RMSE $= s/\sqrt{12}$。下面在真实 GPT-2 权重上把量化残差的实测方差和理论 $s^2/12$ 对一对，验证这个噪声模型成立——也就证明了 **误差正比于 $s^2$，减小步长（per-channel/group）直接减误差**。

In [ ]:
# 量化误差的理论值：round 到步长 s 的误差 ~ 均匀分布 U(-s/2, s/2)，方差 = s²/12。
# 在真实 GPT-2 权重上对比实测误差方差与理论 s²/12。
q, s = quant_symmetric(W)           # 对称 int8，步长 = s
resid = (W - dequant_symmetric(q, s)).ravel()   # 量化残差
var_emp = resid.var()
var_theory = s**2/12
print(f"int8 对称量化 步长 s = {s:.6f}")
print(f"  实测残差方差   = {var_emp:.3e}")
print(f"  理论 s²/12     = {var_theory:.3e}")
print(f"  比值(实测/理论) = {var_emp/var_theory:.3f}  (应接近 1)")
print(f"  理论 RMSE=s/√12 = {s/np.sqrt(12):.6f}  vs 实测 = {resid.std():.6f}")
# 自检：实测方差与 s²/12 同量级（均匀量化噪声模型成立）
assert 0.7 < var_emp/var_theory < 1.3, "量化残差应近似 U(-s/2,s/2)，方差≈s²/12"
print("\n=> 误差方差正比于 s²，所以减小步长 s 直接减小误差——")
print("   per-channel/group 每块用更小的 s，这是它误差更小的数学根源")

---
## ✏️ 练习区

### ✏️ 练习 1：对称 int8 量化/反量化

实现 `sym_quant(W)` 返回 `(q_int8, scale)` 和 `sym_dequant(q, s)`。

In [ ]:
def sym_quant(W):
    # TODO: s = max|W|/127 ; q = round(W/s) 转 int8
    raise NotImplementedError
def sym_dequant(q, s):
    # TODO: q*s
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测（真实 GPT-2 权重）——
q, s = sym_quant(W)
assert q.dtype == np.int8 and np.abs(q).max() <= 127
Wq = sym_dequant(q, s)
rel = np.abs(W-Wq).mean()/np.abs(W).mean()
# 你的实现应与参考一致：per-tensor absmax int8。这块真实权重含离群值，
# per-tensor 误差约 9%（远超课本的 <1%，正是后面 per-channel 要解决的问题）。
qref, sref = quant_symmetric(W)
assert np.array_equal(q, qref) and abs(s - sref) < 1e-12, "应为 per-tensor absmax: s=max|W|/127"
assert rel < 0.10, f"per-tensor int8 这块权重误差约 9%，你的 {rel:.2%}"
print(f"练习 1 通过 ✓  真实GPT-2权重 per-tensor int8 相对误差={rel:.3%}(离群值所致, 见第3节)")


### ✏️ 练习 2：非对称量化

实现 `asym_quant(X)` 返回 `(q_uint8, scale, zero_point)`，把 `[min,max]` 映射到 `[0,255]`。

In [ ]:
def asym_quant(X):
    # TODO: s=(max-min)/255 ; z=round(-min/s) ; q=clip(round(X/s)+z,0,255) uint8
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
X = np.maximum(0, W.ravel()[:1000])   # 全正
q,s,z = asym_quant(X)
assert q.dtype==np.uint8 and q.min()>=0 and q.max()<=255
deq = (q.astype(np.float32)-z)*s
# uint8(256级) 非对称量化全正数据，相对误差约 1%（比 per-tensor int8 整块权重小得多，
# 因为这里数据范围窄、无极端离群值）
assert np.abs(X-deq).mean()/(np.abs(X).mean()+1e-9) < 0.02
# 全正数据 zero-point 应接近 0（min≈0）
assert z < 30
print(f"练习 2 通过 ✓  zero-point={z:.0f}, 相对误差={np.abs(X-deq).mean()/(np.abs(X).mean()+1e-9):.3%}")


### ✏️ 练习 3：per-channel 误差更小

实现 `per_channel_quant(W)`（每列一个 scale），返回 `(q, scales)`。
验证：在真实 GPT-2 权重上，per-channel 误差 < per-tensor。

In [ ]:
def per_channel_quant(W):
    # TODO: scales = max|W|按列/127 (shape (1,cols)) ; q=round(W/scales)
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
qc, sc = per_channel_quant(W)
assert sc.shape == (1, W.shape[1])
err_c = np.abs(W - qc.astype(np.float32)*sc).mean()/np.abs(W).mean()
qt, st = sym_quant(W); err_t = np.abs(W-qt*st).mean()/np.abs(W).mean()
assert err_c < err_t, "per-channel 应比 per-tensor 误差小"
print(f"练习 3 通过 ✓  per-tensor={err_t:.3%} -> per-channel={err_c:.3%}")


### ✏️ 练习 4：保护重要列（AWQ 思想简化版）

实现 `protected_quant(W, keep_frac)`：把"L2 范数最大的 `keep_frac` 比例的列"保留 fp32，其余 int8。
返回反量化后的矩阵。验证：保护后误差 < 全 int8。

In [ ]:
def protected_quant(W, keep_frac=0.01):
    # TODO: 算每列 L2 范数，选最大的 keep_frac 比例列保 fp32；其余列对称 int8 量化后反量化；拼回
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
Wp = protected_quant(W, keep_frac=0.05)
err_p = np.abs(W-Wp).mean()/np.abs(W).mean()
qt, st = sym_quant(W); err_full = np.abs(W-qt*st).mean()/np.abs(W).mean()
assert err_p < err_full, "保护重要列后误差应下降"
assert Wp.shape == W.shape
print(f"练习 4 通过 ✓  全int8={err_full:.3%} -> 保护5%列={err_p:.3%}（AWQ 核心直觉）")


---
## 📖 参考答案

In [ ]:
# 练习 1
def sym_quant(W):
    s=np.abs(W).max()/127; return np.round(W/s).astype(np.int8), s
def sym_dequant(q, s): return q.astype(np.float32)*s
print("练习 1 ✓")

In [ ]:
# 练习 2
def asym_quant(X):
    lo,hi=X.min(),X.max(); s=(hi-lo)/255; z=np.round(-lo/s)
    return np.clip(np.round(X/s)+z,0,255).astype(np.uint8), s, z
print("练习 2 ✓")

In [ ]:
# 练习 3
def per_channel_quant(W):
    s=np.abs(W).max(0,keepdims=True)/127; return np.round(W/s).astype(np.int8), s
print("练习 3 ✓")

In [ ]:
# 练习 4
def protected_quant(W, keep_frac=0.01):
    col_norm=np.linalg.norm(W,axis=0); k=max(1,int(keep_frac*W.shape[1]))
    keep=set(np.argsort(-col_norm)[:k])
    out=W.copy()
    for j in range(W.shape[1]):
        if j in keep: continue
        col=W[:,j]; s=np.abs(col).max()/127+1e-12
        out[:,j]=np.round(col/s).astype(np.int8)*s
    return out
print("练习 4 ✓ —— 误差不该被均匀对待，保护重要权重是 GPTQ/AWQ 的灵魂")